<a href="https://colab.research.google.com/github/Kailash-192524029/DSA0404-FoDS/blob/main/Q20_AB_Test_Conversion_Rates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q20 - A/B Test: Website Design A vs B

**Question:** Is there a statistically significant difference in the mean conversion rates between design A and design B?

**Dataset:** `20_AB_Testing_Website_Dataset.csv` (columns `Design` and `Converted`, where 1 = converted).

Run the cells from top to bottom. The first code cell asks you to upload the dataset.

In [ ]:
# Step 1: Upload the dataset (a file picker will appear - choose 20_AB_Testing_Website_Dataset.csv)
from google.colab import files
uploaded = files.upload()
file_name = next(iter(uploaded))
print("Loaded file:", file_name)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_csv(file_name)
print(df.shape)
df.head()

## Step 2: Conversion rate of each design

In [ ]:
summary = df.groupby("Design")["Converted"].agg(Visitors="count", Conversions="sum", Conversion_Rate="mean")
print(summary)

a = df[df["Design"] == "A"]["Converted"]
b = df[df["Design"] == "B"]["Converted"]

summary["Conversion_Rate"].plot(kind="bar", color=["steelblue", "seagreen"], rot=0, figsize=(5, 4))
plt.title("Conversion rate by design"); plt.ylabel("Conversion rate")
plt.show()

## Step 3: Hypothesis test
H0: mean conversion rates of A and B are equal.  H1: they are different.  Significance level = 0.05

In [ ]:
alpha = 0.05

# Two-sample t-test (Welch) on the 0/1 conversion values
t_stat, p_value = stats.ttest_ind(a, b, equal_var=False)
print(f"T-test:  t = {t_stat:.4f}, p-value = {p_value:.6f}")

# Two-proportion z-test (same idea, designed for conversion rates)
p1, p2 = a.mean(), b.mean()
n1, n2 = len(a), len(b)
p_pool = (a.sum() + b.sum()) / (n1 + n2)
se = np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
z = (p1 - p2) / se
p_z = 2 * (1 - stats.norm.cdf(abs(z)))
print(f"Z-test:  z = {z:.4f}, p-value = {p_z:.6f}")

print("\nConversion rate A:", round(p1, 4), "| B:", round(p2, 4))
if p_value < alpha:
    better = "B" if p2 > p1 else "A"
    print(f"Result: p-value < {alpha} -> statistically SIGNIFICANT difference. Design {better} converts better.")
else:
    print(f"Result: p-value >= {alpha} -> NO significant difference between the designs.")